# 03 — HDMI trực tiếp → FPGA → Jupyter / màn hình tùy chọn

Notebook chạy trên **PYNQ-Z2**, dùng overlay `video_hdmi`. Không cần làm inventory riêng. Nên chạy notebook 01 PASS trước để xác nhận bộ lọc/DMA với ảnh kiểm thử.

**Chưa có màn hình ngoài vẫn chạy được:** nối một cáp từ **HDMI OUT laptop → HDMI IN của board**. Jupyter vẫn mở trên laptop qua Ethernet. Trên Windows chọn **Win+P → Duplicate**, đặt tín hiệu HDMI **1280×720, 60 Hz** trong Settings → Display → Advanced display. Đây là chế độ EDID mặc định, phù hợp clock RX của thiết kế. Không chọn đầu vào HDMI VGA; pipeline RX hiện tại không bảo đảm bắt được pixel clock VGA. Nếu driver báo 1920×1080, đổi đầu ra Windows về 720p rồi chạy lại cell.

Đường đi: HDMI IN 1280×720 → VDMA/DDR → **ARM resize về 640×480** → AXI DMA → **Verilog bộ lọc** → AXI DMA/DDR. Resize dùng nearest-neighbour NumPy, giữ tỷ lệ 16:9: nội dung 640×360 được thêm 60 hàng đen phía trên/dưới. Resize và preview chạy trên ARM, cả bốn bộ lọc chạy trên FPGA. Thời gian resize được đo riêng. Mặc định **không bật HDMI OUT** vì chưa có màn hình ngoài.

Upload **`START_HERE.ipynb`** và **`video_pynq_z2_deploy.zip`** vào cùng thư mục Jupyter của board. Chạy **Run All** trong `START_HERE.ipynb` để kiểm tra SHA256 và giải nén, không cần Terminal. Mở notebook này trong `video_demo`, dừng notebook/ứng dụng MNIST hoặc DMA khác, chọn **Kernel → Restart & Run All**. Khi đổi overlay, phải dừng DMA của notebook cũ trước.


In [ ]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
if not (ROOT / 'video_driver.py').is_file():
    raise RuntimeError('Mở notebook trong thư mục video_demo đã giải nén trên board, có video_driver.py cạnh notebook.')
sys.path.insert(0, str(ROOT))
RESULTS = ROOT / 'results'
RESULTS.mkdir(exist_ok=True)
print('Thư mục demo:', ROOT)


In [ ]:
from board_inventory import collect_inventory
inventory = collect_inventory()
(RESULTS / 'board_inventory_hdmi.json').write_text(json.dumps(inventory, indent=2, ensure_ascii=False), encoding='utf-8')
print('PYNQ:', inventory['packages']['pynq'], '| Board:', inventory['board_model'] or inventory['board_env'])


## Chọn bộ lọc và giới hạn lần chạy

Mặc định xử lý tối đa 120 frame rồi tự dừng, giải phóng buffer. Chạy lại cell thực thi để xem tiếp. Đổi bộ lọc trong menu khi vòng lặp đang chạy; thay đổi chỉ áp dụng từ frame tiếp theo. Nếu image không có widget, sửa `MODE`/`THRESHOLD` rồi chạy lại.

`PROCESSING_SIZE=(640,480)` là mặc định xử lý VGA từ nguồn HDMI 720p. Đổi thành `None` để xử lý đầy đủ 1280×720; độ trễ và tải DDR tăng, Sobel không được cam kết 30 FPS. Báo cáo lưu riêng kích thước nguồn và kích thước xử lý.

Chỉ đặt `ENABLE_HDMI_OUTPUT=True` sau khi có cáp từ **HDMI OUT board → HDMI IN màn hình/TV**. Cổng HDMI laptop không dùng để nhận đầu ra board. Khi có màn hình, nội dung mới được cập nhật theo tốc độ xử lý thực tế; màn hình có thể lặp frame để duy trì refresh 60 Hz.


In [ ]:
ENABLE_HDMI_OUTPUT = False
PROCESSING_SIZE = (640, 480)  # ARM resize + black padding; None: FPGA xử lý đủ 720p
FRAMES = 120
MODE = 3                    # 0: gốc, 1: xám, 2: ngưỡng, 3: Sobel
THRESHOLD = 128
PREVIEW_EVERY = 10           # Tăng để giảm tải Jupyter
VERIFY_EVERY = 30            # CPU chỉ kiểm chứng vài frame, không thay output FPGA
FRAME_TIMEOUT = 3.0
DETECT_TIMEOUT = 15.0        # Thiếu tín hiệu HDMI sẽ báo lỗi sau khoảng này

from reference_filters import MODE_NAMES
mode_control = threshold_control = None
try:
    import ipywidgets as widgets
except ImportError:
    print('Không có widget: dùng MODE/THRESHOLD ở đầu cell.')
else:
    from IPython.display import display
    mode_control = widgets.Dropdown(options=[(MODE_NAMES[m],m) for m in range(4)], value=MODE, description='Bộ lọc')
    threshold_control = widgets.IntSlider(value=THRESHOLD,min=0,max=255,description='Ngưỡng',continuous_update=False)
    display(widgets.HBox([mode_control, threshold_control]))

def current_controls(index):
    return (MODE, THRESHOLD) if mode_control is None else (mode_control.value, threshold_control.value)


## Chạy demo

Cell dùng `await` của Jupyter để đọc/ghi frame có timeout, cho phép menu cập nhật. Không dùng `asyncio.run()` trong notebook. Có một frame đầu được bỏ để tránh buffer cũ; mỗi frame sau được đưa qua bộ lọc FPGA.

Preview được thu nhỏ và mã hóa JPEG trên ARM, nên nhịp hình trong trình duyệt chậm hơn đường HDMI. Báo cáo không giả định số frame nguồn bị bỏ bằng 0: VDMA không cung cấp sequence ID của nguồn cho luồng này.


In [ ]:
import base64, io
import numpy as np
from PIL import Image
from IPython.display import display, HTML
from reference_filters import rgb_for_display
from hdmi_workflows import run_hdmi_demo

preview_handle = display(HTML('Đang chờ tín hiệu HDMI...'), display_id=True)

def jpeg_data(bgr):
    image = Image.fromarray(rgb_for_display(bgr))
    image.thumbnail((480, 270))
    stream = io.BytesIO()
    image.save(stream, format='JPEG', quality=80)
    return 'data:image/jpeg;base64,' + base64.b64encode(stream.getvalue()).decode('ascii')

def show_hdmi_preview(source, actual, index, metrics):
    mode = int(metrics['mode'])
    pictures = []
    input_label = 'Đầu vào bộ lọc sau resize ARM (%dx%d)' % (source.shape[1], source.shape[0])
    for label, image in [(input_label, source), ('Output FPGA — ' + MODE_NAMES[mode], actual)]:
        pictures.append('<figure style="margin:8px"><figcaption>' + label + '</figcaption><img style="max-width:100%" src="' + jpeg_data(image) + '"></figure>')
    text = 'Frame %d | resize ARM %.3f ms | core %.3f ms | driver %.3f ms | toàn frame chưa preview %.3f ms' % (
        index, metrics['arm_resize_ms'], metrics['core_active_ms'], metrics['total_driver_ms'], metrics['whole_frame_without_preview_ms'])
    preview_handle.update(HTML('<div style="display:flex;flex-wrap:wrap">' + ''.join(pictures) + '</div><p>' + text + '</p>'))

hdmi_report = await run_hdmi_demo(ROOT,
    enable_output=ENABLE_HDMI_OUTPUT, detect_timeout=DETECT_TIMEOUT, processing_size=PROCESSING_SIZE,
    frames=FRAMES, mode=MODE, threshold=THRESHOLD,
    frame_timeout=FRAME_TIMEOUT, controls=current_controls,
    preview=show_hdmi_preview, preview_every=PREVIEW_EVERY, verify_every=VERIFY_EVERY)
print('HDMI FPGA:', hdmi_report['status'], '| Frame:', hdmi_report['frames_processed'])
print('Nguồn HDMI: %dx%d | FPGA xử lý: %dx%d' % (hdmi_report['source_width'],hdmi_report['source_height'],hdmi_report['processing_width'],hdmi_report['processing_height']))
print('Nhịp xử lý quan sát gồm preview: %.2f frame/s' % hdmi_report['observed_processed_frames_per_second_including_preview'])
print('Đối chiếu pixel:', len(hdmi_report['pixel_checks']), '| Driver trung bình: %.3f ms' % hdmi_report['driver_time']['mean_ms'])
print('Resize ARM trung bình %.3f ms | Cả frame không gồm preview %.3f ms' % (hdmi_report['arm_resize_time']['mean_ms'],hdmi_report['whole_frame_without_preview']['mean_ms']))
print('Nguồn bị bỏ bao nhiêu frame: chưa đo; tốc độ render thật của trình duyệt: chưa đo.')
print('Đã dừng VDMA/AXI DMA. Báo cáo:', RESULTS / 'hdmi_results.json')


## Đọc kết quả và xử lý lỗi

- `results/hdmi_results.json`: kích thước nguồn/thực sự xử lý, mode từng frame, thời gian resize ARM, chu kỳ core, thời gian driver/cả frame, nhịp vòng lặp gồm preview và kết quả so pixel mẫu.
- **PASS** xác nhận các frame đã đọc được qua FPGA và các frame kiểm chứng khớp; không chứng minh mọi frame HDMI của nguồn đều được xử lý, không tự xác nhận hình trên màn hình ngoài đẹp/ổn định.
- Timeout đầu vào: kiểm tra đúng **HDMI IN**, dây cáp và Windows đang xuất **1280×720, 60 Hz**. Không chọn VGA cho đầu vào HDMI với cấu hình RX hiện tại. Cắm lại nguồn video rồi chạy lại cell thực thi.
- Lỗi binding hierarchy: gửi `results/board_inventory_hdmi.json` và traceback; image PYNQ khác phiên bản có thể cần điều chỉnh driver.
- Lỗi timeout khi dừng VDMA hoặc reset DMA: **khởi động lại board trước khi nạp overlay khác**. Không giải phóng vùng nhớ mà DMA có thể còn sử dụng.

Notebook tự đóng DMA khi hoàn tất, lỗi hoặc ngắt cell. Khi thử đầu ra với màn hình sau này, em cần tự xác nhận màu, hướng ảnh, biên, chuyển mode và độ ổn định; lưu thêm video demo làm bằng chứng vật lý.
